In [54]:
import json
import re
import time
import math
import hashlib

from pathlib import Path
from typing import List, Dict, Any
from dotenv import load_dotenv
from openai import OpenAI

In [55]:
load_dotenv()
client = OpenAI()

In [56]:
MODEL_NAME = "gpt-4.1-mini"

INPUT_FILES = [
    "./data/fia_2026_f1_regulations_-_section_a_general_provisions_-_iss_02_-_2026-02-27.md",
    "./data/fia_2026_f1_regulations_-_section_b_sporting_-_iss_05_-_2026-02-27 (1).md",
]

# 최종 목표 개수
TARGET_QA_A = 2000
TARGET_QA_B = 2000

# 중복 제거 감안해서 약간 더 생성
BUFFER_RATIO = 1.30

ARTICLE_LIMIT_A = 2500
ARTICLE_LIMIT_B = 2500

SLEEP_SECONDS = 0.5
MAX_RETRIES = 3

# 추가 설정
BATCH_SIZE = 10                 # 한 번에 생성 요청할 QA 수
MIN_SUCCESS_RATIO = 0.7         # 요청 개수 대비 최소 허용 비율
MAX_SECTION_ROUNDS = 20         # 섹션별 최대 반복 라운드 수
RECENT_QUESTION_MEMORY = 12     # 같은 조문에서 최근 질문 몇 개를 중복 방지용으로 넘길지

OUTPUT_DIR = Path("output_qa_pipeline_md")
OUTPUT_DIR.mkdir(exist_ok=True)

In [57]:
def get_run_paths():
    run_tag = f"A{TARGET_QA_A}_B{TARGET_QA_B}"
    finetune_jsonl_path = OUTPUT_DIR / f"f1_finetune_qa_{run_tag}.jsonl"
    pretty_json_path = OUTPUT_DIR / f"f1_finetune_qa_{run_tag}.json"
    raw_json_path = OUTPUT_DIR / f"raw_qa_{run_tag}.json"
    return finetune_jsonl_path, pretty_json_path, raw_json_path

In [58]:
def read_markdown(path: str) -> str:
    with open(path, "r", encoding="utf-8") as f:
        return f.read()


def load_multiple_markdowns(paths: List[str]) -> List[Dict[str, Any]]:
    docs = []

    for path in paths:
        text = read_markdown(path)
        docs.append({
            "source_file": Path(path).name,
            "text": text
        })
    
    return docs

In [59]:
def normalize_text(text: str) -> str:
    text = text.replace("\u00ad", "")
    text = text.replace("\xa0", " ")
    text = text.replace("−", "-")
    text = text.replace("–", "-")
    text = re.sub(r"\r\n?", "\n", text)
    text = re.sub(r"\n{3,}", "\n\n", text)
    return text.strip()


def keep_main_regulation_body(text: str) -> str:
    """
    본문 시작: ARTICLE A1 / ARTICLE B1 이후
    본문 종료: 첫 APPENDIX 이전
    """
    start_match = re.search(r'(?m)^##\s*\*\*ARTICLE\s+[A-Z]1\s*:', text)
    if start_match:
        text = text[start_match.start():]

    appendix_match = re.search(r'(?m)^##\s*\*\*APPENDIX\s+', text)
    if appendix_match:
        text = text[:appendix_match.start()]

    return text.strip()


def clean_article_text(text: str) -> str:
    # markdown 장식 제거
    text = text.replace("**", "")
    text = text.replace("_", "")

    # 빈 불릿 제거
    text = re.sub(r"(?m)^\s*-\s*$", "", text)

    # 페이지 헤더/푸터 제거
    text = re.sub(r"(?m)^2026 Formula 1:.*$", "", text)
    text = re.sub(r"(?m)^©2026 Fédération Internationale de l’Automobile$", "", text)

    # 과한 줄바꿈 정리
    text = re.sub(r"\n{3,}", "\n\n", text)
    return text.strip()

In [60]:
def article_sort_key(article_id: str):
    nums = re.findall(r"\d+", article_id)
    return tuple(int(x) for x in nums)


def parse_level2_heading(line: str):
    """
    예:
    ## **A1.1 Overview**
    ## **A2.2 Championship points system**
    ## **B1.5 General Safety**
    """
    line = line.strip()
    m = re.match(r'^##\s*\*\*([A-Z]\d+\.\d+)\s+(.+?)\*\*\s*$', line)
    if not m:
        return None

    article_id = m.group(1).strip()
    title = re.sub(r"\s+", " ", m.group(2)).strip()
    return article_id, title


def split_articles(text: str, source_file: str) -> List[Dict[str, Any]]:
    text = normalize_text(text)
    text = keep_main_regulation_body(text)
    lines = text.split("\n")

    articles = []
    current = None
    buffer = []

    def flush():
        nonlocal current, buffer, articles

        if current is None:
            return

        article_text = "\n".join(buffer).strip()
        article_text = clean_article_text(article_text)

        # 너무 짧은 조항 제거
        if len(article_text) < 120:
            return

        # 내용이 거의 없는 경우 제거
        if article_text.count("\n") < 1:
            return

        articles.append({
            "source_file": source_file,
            "article_id": current["article_id"],
            "article_title": current["article_title"],
            "article_text": article_text
        })

    for raw_line in lines:
        parsed = parse_level2_heading(raw_line)

        if parsed:
            flush()
            article_id, article_title = parsed
            current = {
                "article_id": article_id,
                "article_title": article_title
            }
            buffer = [raw_line]
            continue

        if current is not None:
            buffer.append(raw_line)

    flush()
    articles.sort(key=lambda x: article_sort_key(x["article_id"]))
    return articles

In [61]:
def get_section_key(source_file: str) -> str:
    name = source_file.lower()

    if "section_a" in name:
        return "A"
    if "section_b" in name:
        return "B"

    raise ValueError(f"섹션을 판별할 수 없는 파일명입니다: {source_file}")


def group_articles_by_section(all_articles: List[Dict[str, Any]]) -> Dict[str, List[Dict[str, Any]]]:
    grouped = {"A": [], "B": []}

    for article in all_articles:
        section = get_section_key(article["source_file"])
        grouped[section].append(article)

    return grouped


def sample_articles_evenly(articles: List[Dict[str, Any]], limit: int) -> List[Dict[str, Any]]:
    if len(articles) <= limit:
        return articles

    step = len(articles) / limit
    indices = []
    for i in range(limit):
        idx = min(len(articles) - 1, int(round(i * step)))
        if idx not in indices:
            indices.append(idx)

    return [articles[i] for i in indices]


def select_articles(grouped_articles: Dict[str, List[Dict[str, Any]]]) -> Dict[str, List[Dict[str, Any]]]:
    return {
        "A": sample_articles_evenly(grouped_articles["A"], ARTICLE_LIMIT_A),
        "B": sample_articles_evenly(grouped_articles["B"], ARTICLE_LIMIT_B),
    }

In [62]:
docs = load_multiple_markdowns(INPUT_FILES)

all_articles = []
for doc in docs:
    split_result = split_articles(doc["text"], doc["source_file"])
    print(doc["source_file"], "->", len(split_result), "개")
    all_articles.extend(split_result)

print("총 조문 수:", len(all_articles))
print()

for a in all_articles[:20]:
    print(f"{a['article_id']} | {a['article_title']}")
    print(a["article_text"][:300])
    print("-" * 100)

fia_2026_f1_regulations_-_section_a_general_provisions_-_iss_02_-_2026-02-27.md -> 37 개
fia_2026_f1_regulations_-_section_b_sporting_-_iss_05_-_2026-02-27 (1).md -> 48 개
총 조문 수: 85

A1.1 | Overview
## A1.1 Overview 

- A1.1.1 The FIA is responsible for the sporting organisation and regulation of the FIA Formula One World Championship (the “ Championship ”), comprising the FIA Formula One Grand Prix competitions that are included on the International Sporting Calendar (each, a “ Competition ”),
----------------------------------------------------------------------------------------------------
A1.3 | Application
## A1.3 Application 

- A1.3.1 any Competition, or any F1 Activities, is deemed, as a condition of such participation, to have agreed to be bound by and to comply with the FIA Rules and Regulations (including the FIA F1 Regulations) and all Decisions, and to have submitted to the authority of the FI
--------------------------------------------------------------------------------

In [63]:
BATCH_FOCUS_ORDER = [
    "개념형",
    "요약형",
    "수치/기준형",
    "오해 교정형",
    "비교형",
]

def build_prompt(
    article: Dict[str, Any],
    n_questions: int,
    batch_focus: str = "균형형",
    recent_questions: List[str] = None
) -> str:
    recent_questions = recent_questions or []

    recent_block = ""
    if recent_questions:
        recent_lines = "\n".join([f"- {q}" for q in recent_questions[-RECENT_QUESTION_MEMORY:]])
        recent_block = f"""
[이미 생성된 최근 질문들]
아래 질문들과 의미가 겹치지 않게 새롭게 만들어라:
{recent_lines}
""".strip()

    return f"""
[조문 정보]
- 문서: {article["source_file"]}
- 조항 ID: {article["article_id"]}
- 조항 제목: {article["article_title"]}

[조문 원문]
{article["article_text"]}

[이번 배치 생성 개수]
총 {n_questions}개

[이번 배치 중점 유형]
{batch_focus}

[생성 규칙]
- 조문 내용만 바탕으로 작성
- 한국어로 작성
- 초보자도 이해하기 쉽게 설명
- 답변은 2~4문장
- 숫자, 단위(mm, ms, 개수, 각도)는 조문 기준 그대로 유지
- 규정 원문에 없는 내용 단정 금지
- 질문끼리 의미 중복이 크지 않게 작성
- 단순 문장 재진술만 하지 말고, 입문자 질문처럼 자연스럽게 만들 것
- 조항 번호만 언급하지 말고, 내용을 풀어서 설명할 것
- 오해 교정형은 왜 틀렸는지도 설명할 것
- 비교형은 같은 조문 안에서 비교 가능한 대상이 있을 때만 만들고,
  비교가 어렵다면 억지로 만들지 말고 다른 허용 유형으로 대체할 것
- 이번 배치에서는 특히 "{batch_focus}" 비중을 높이되,
  질문 문형과 초점은 서로 다르게 만들 것
- 반드시 서로 다른 질문 {n_questions}개를 만들 것
- 출력은 JSON 배열만 출력할 것
- 코드블록(```)을 쓰지 말 것
- 배열 밖의 설명 문장은 절대 쓰지 말 것

{recent_block}

[출력 형식]
반드시 아래 JSON 배열만 출력:
[
  {{
    "question": "...",
    "answer": "..."
  }}
]
""".strip()


def extract_json_array(text: str) -> str:
    match = re.search(r"\[[\s\S]*\]", text)
    if not match:
        raise ValueError("JSON 배열을 찾지 못했습니다.")
    return match.group(0)

In [64]:
def generate_qa(
    article: Dict[str, Any],
    n_questions: int,
    batch_focus: str = "균형형",
    recent_questions: List[str] = None
) -> List[Dict[str, Any]]:
    prompt = build_prompt(
        article=article,
        n_questions=n_questions,
        batch_focus=batch_focus,
        recent_questions=recent_questions or []
    )

    min_expected = max(1, math.ceil(n_questions * MIN_SUCCESS_RATIO))

    for attempt in range(MAX_RETRIES):
        try:
            response = client.responses.create(
                model=MODEL_NAME,
                input=prompt
            )

            text = response.output_text.strip()

            try:
                data = json.loads(text)
            except json.JSONDecodeError:
                data = json.loads(extract_json_array(text))

            if not isinstance(data, list):
                raise ValueError("모델 출력이 JSON 배열이 아닙니다.")

            results = []
            for item in data:
                if not isinstance(item, dict):
                    continue
                if "question" not in item or "answer" not in item:
                    continue

                question = str(item["question"]).strip()
                answer = str(item["answer"]).strip()

                if not question or not answer:
                    continue

                results.append({
                    "source_file": article["source_file"],
                    "article_id": article["article_id"],
                    "article_title": article["article_title"],
                    "question": question,
                    "answer": answer
                })

            # 같은 응답 안에서 1차 중복 제거
            results = deduplicate_qa(results)

            print(
                f"    요청 {n_questions}개 | 실제 반환 {len(results)}개 | "
                f"유형 {batch_focus} | 시도 {attempt + 1}/{MAX_RETRIES}"
            )

            if len(results) < min_expected:
                raise ValueError(
                    f"생성 수 부족: 요청 {n_questions}개, 실제 {len(results)}개"
                )

            return results

        except Exception as e:
            print(f"    [재시도 {attempt + 1}/{MAX_RETRIES}] {article['article_id']} 실패: {e}")
            time.sleep(1.5)

    return []

In [65]:
def normalize_qa_text(text: str) -> str:
    text = text.strip()
    text = re.sub(r"\s+", " ", text)
    text = text.replace("“", '"').replace("”", '"')
    text = text.replace("‘", "'").replace("’", "'")
    return text


def qa_hash(item: Dict[str, Any]) -> str:
    q = normalize_qa_text(item["question"])
    a = normalize_qa_text(item["answer"])
    key = f"{q} || {a}"
    return hashlib.md5(key.encode("utf-8")).hexdigest()


def deduplicate_qa(items: List[Dict[str, Any]]) -> List[Dict[str, Any]]:
    seen = set()
    deduped = []

    for item in items:
        h = qa_hash(item)
        if h in seen:
            continue
        seen.add(h)
        deduped.append(item)

    return deduped


def get_recent_questions_for_article(
    items: List[Dict[str, Any]],
    article_id: str,
    limit: int = RECENT_QUESTION_MEMORY
) -> List[str]:
    questions = [
        x["question"]
        for x in items
        if x["article_id"] == article_id
    ]
    return questions[-limit:]


def get_batch_focus(round_idx: int, article_idx: int) -> str:
    return BATCH_FOCUS_ORDER[(round_idx + article_idx) % len(BATCH_FOCUS_ORDER)]

In [66]:
def to_finetune_messages(item: Dict[str, Any]) -> Dict[str, Any]:
    return {
        "messages": [
            {
                "role": "user",
                "content": item["question"]
            },
            {
                "role": "assistant",
                "content": item["answer"]
            }
        ]
    }

In [67]:
def save_jsonl(path: Path, rows: List[Dict[str, Any]]):
    with open(path, "w", encoding="utf-8") as f:
        for row in rows:
            f.write(json.dumps(row, ensure_ascii=False) + "\n")


def save_pretty_json(path: Path, rows: Any):
    with open(path, "w", encoding="utf-8") as f:
        json.dump(rows, f, ensure_ascii=False, indent=2)


def save_raw_json(path: Path, rows: Any):
    with open(path, "w", encoding="utf-8") as f:
        json.dump(rows, f, ensure_ascii=False, indent=2)

In [68]:
def get_generation_targets():
    section_targets = {
        "A": TARGET_QA_A,
        "B": TARGET_QA_B,
    }

    section_generation_targets = {
        sec: math.ceil(target * BUFFER_RATIO)
        for sec, target in section_targets.items()
    }

    return section_targets, section_generation_targets

In [69]:
def main():
    print("[1] MD 불러오기")
    docs = load_multiple_markdowns(INPUT_FILES)

    print("[2] 조문 분리")
    all_articles = []
    for doc in docs:
        split_result = split_articles(doc["text"], doc["source_file"])
        print(f"  - {doc['source_file']} -> {len(split_result)}개")
        all_articles.extend(split_result)

    print(f"전체 조문 수: {len(all_articles)}")

    if len(all_articles) == 0:
        raise ValueError("조문이 0개입니다. split_articles() 결과를 먼저 확인하세요.")

    grouped_articles = group_articles_by_section(all_articles)
    selected_grouped_articles = select_articles(grouped_articles)

    section_targets, section_generation_targets = get_generation_targets()
    finetune_jsonl_path, pretty_json_path, raw_json_path = get_run_paths()

    print("\n[3] 실행 설정")
    print("BUFFER_RATIO:", BUFFER_RATIO)
    print("BATCH_SIZE:", BATCH_SIZE)
    print("MIN_SUCCESS_RATIO:", MIN_SUCCESS_RATIO)
    print("MAX_SECTION_ROUNDS:", MAX_SECTION_ROUNDS)
    print("Section별 최종 목표:", section_targets)
    print("Section별 생성 목표(버퍼 반영):", section_generation_targets)
    print("Section별 전체 조문 수:", {k: len(v) for k, v in grouped_articles.items()})
    print("Section별 이번 실행 조문 수:", {k: len(v) for k, v in selected_grouped_articles.items()})

    print("\n[실행 대상 조문]")
    for sec in ["A", "B"]:
        print(f"[Section {sec}]")
        for a in selected_grouped_articles[sec][:20]:
            print(f"  - {a['article_id']} | {a['article_title']}")

    print("\n[4] QA 생성")

    all_final_qa = []
    all_raw_qa = []
    section_stats = {}

    for sec in ["A", "B"]:
        articles_sec = selected_grouped_articles[sec]

        if len(articles_sec) == 0:
            print(f"\n[Section {sec}] 조문이 0개라서 건너뜀")
            section_stats[sec] = {
                "raw_count": 0,
                "deduped_count": 0,
                "final_count": 0,
                "rounds": 0
            }
            continue

        target_final = section_targets[sec]
        target_generation = section_generation_targets[sec]

        print(f"\n[Section {sec}]")
        print(f"  최종 목표: {target_final}")
        print(f"  생성 목표(버퍼 반영): {target_generation}")
        print(f"  조문 수: {len(articles_sec)}")
        print(f"  배치 크기: {BATCH_SIZE}")

        section_raw = []
        section_rounds = 0

        while section_rounds < MAX_SECTION_ROUNDS:
            section_rounds += 1

            section_deduped_before = deduplicate_qa(section_raw)
            before_count = len(section_deduped_before)

            print(f"\n  [Section {sec} | Round {section_rounds}] 시작")
            print(f"    현재 dedup 기준 누적: {before_count} / {target_final}")

            if before_count >= target_final:
                print(f"    이미 목표 달성. Round 종료")
                break

            for i, article in enumerate(articles_sec, start=1):
                current_deduped = deduplicate_qa(section_raw)
                current_final_count = len(current_deduped)

                if current_final_count >= target_final:
                    print(f"    목표 도달로 Section {sec} 생성 중단")
                    break

                need = target_final - current_final_count
                n_questions = min(BATCH_SIZE, need)
                batch_focus = get_batch_focus(section_rounds, i)
                recent_questions = get_recent_questions_for_article(
                    section_raw,
                    article["article_id"],
                    limit=RECENT_QUESTION_MEMORY
                )

                print(
                    f"  - ({i}/{len(articles_sec)}) {article['article_id']} | {article['article_title']}"
                )
                print(
                    f"    필요 잔여량: {need} | 이번 배치 요청: {n_questions} | 유형: {batch_focus}"
                )

                qa_items = generate_qa(
                    article=article,
                    n_questions=n_questions,
                    batch_focus=batch_focus,
                    recent_questions=recent_questions
                )

                if qa_items:
                    section_raw.extend(qa_items)
                    all_raw_qa.extend(qa_items)

                current_after_raw = len(section_raw)
                current_after_dedup = len(deduplicate_qa(section_raw))

                print(f"    Section {sec} raw 누적: {current_after_raw}")
                print(f"    Section {sec} dedup 누적: {current_after_dedup}")

                time.sleep(SLEEP_SECONDS)

            section_deduped_after = deduplicate_qa(section_raw)
            after_count = len(section_deduped_after)

            print(f"\n  [Section {sec} | Round {section_rounds}] 종료")
            print(f"    round 전 dedup 수: {before_count}")
            print(f"    round 후 dedup 수: {after_count}")
            print(f"    증가량: {after_count - before_count}")

            if after_count >= target_final:
                print(f"    Section {sec} 목표 달성")
                break

            if after_count == before_count:
                print(f"    더 이상 증가하지 않아 Section {sec} 조기 종료")
                break

        section_deduped = deduplicate_qa(section_raw)
        section_final = section_deduped[:target_final]

        print(f"\n  Section {sec} 최종 통계")
        print(f"  - raw 생성 수: {len(section_raw)}")
        print(f"  - dedup 후 수: {len(section_deduped)}")
        print(f"  - 최종 사용 수: {len(section_final)}")

        all_final_qa.extend(section_final)

        section_stats[sec] = {
            "raw_count": len(section_raw),
            "deduped_count": len(section_deduped),
            "final_count": len(section_final),
            "target_final": target_final,
            "target_generation": target_generation,
            "rounds": section_rounds,
            "article_count": len(articles_sec),
        }

    print(f"\n전체 최종 QA 수: {len(all_final_qa)}")

    print("[5] 저장")
    finetune_rows = [to_finetune_messages(item) for item in all_final_qa]

    save_jsonl(finetune_jsonl_path, finetune_rows)

    save_pretty_json(pretty_json_path, {
        "section_stats": section_stats,
        "final_qa": all_final_qa,
        "finetune_rows": finetune_rows,
    })

    save_raw_json(raw_json_path, {
        "section_stats": section_stats,
        "raw_qa": all_raw_qa,
        "final_qa": all_final_qa,
    })

    print("\n완료")
    print(f"- jsonl: {finetune_jsonl_path}")
    print(f"- pretty json: {pretty_json_path}")
    print(f"- raw qa json: {raw_json_path}")

    return {
        "all_articles": all_articles,
        "grouped_articles": grouped_articles,
        "selected_grouped_articles": selected_grouped_articles,
        "section_stats": section_stats,
        "raw_qa": all_raw_qa,
        "final_qa": all_final_qa,
        "jsonl_path": finetune_jsonl_path,
        "pretty_json_path": pretty_json_path,
        "raw_json_path": raw_json_path
    }

In [70]:
result=main()

[1] MD 불러오기
[2] 조문 분리
  - fia_2026_f1_regulations_-_section_a_general_provisions_-_iss_02_-_2026-02-27.md -> 37개
  - fia_2026_f1_regulations_-_section_b_sporting_-_iss_05_-_2026-02-27 (1).md -> 48개
전체 조문 수: 85

[3] 실행 설정
BUFFER_RATIO: 1.3
BATCH_SIZE: 10
MIN_SUCCESS_RATIO: 0.7
MAX_SECTION_ROUNDS: 20
Section별 최종 목표: {'A': 2000, 'B': 2000}
Section별 생성 목표(버퍼 반영): {'A': 2600, 'B': 2600}
Section별 전체 조문 수: {'A': 37, 'B': 48}
Section별 이번 실행 조문 수: {'A': 37, 'B': 48}

[실행 대상 조문]
[Section A]
  - A1.1 | Overview
  - A1.3 | Application
  - A2.1 | Format of the Championship
  - A2.2 | Championship points system
  - A2.3 | F1 Car livery
  - A2.5 | Postponement or cancellation of a Competition
  - A2.6 | FIA Prize Giving Ceremony and Gala Dinner
  - A3.1 | F1 Team entry applications
  - A3.2 | Power Unit Manufacturer entry applications
  - A3.3 | Licences
  - A3.5 | New Entrant Teams
  - A3.6 | New Entrant PU Manufacturers
  - A4.1 | Fit and Proper Persons Test
  - A4.2 | Anti-doping
  - A4.3 | Preven

In [71]:
print("최종 샘플 수:", len(result["final_qa"]))

count_a = sum(1 for x in result["final_qa"] if get_section_key(x["source_file"]) == "A")
count_b = sum(1 for x in result["final_qa"] if get_section_key(x["source_file"]) == "B")

print("Section A 개수:", count_a)
print("Section B 개수:", count_b)
print()

for item in result["final_qa"][:10]:
    print("[Section]", get_section_key(item["source_file"]))
    print("[조문]", item["article_id"], "|", item["article_title"])
    print("Q:", item["question"])
    print("A:", item["answer"])
    print("-" * 100)

최종 샘플 수: 4000
Section A 개수: 2000
Section B 개수: 2000

[Section] A
[조문] A1.1 | Overview
Q: FIA가 주최하는 포뮬러 원 챔피언십 대회의 공식 명칭과 그 구성 요소는 무엇인가요?
A: FIA가 주관하고 규제하는 대회는 'FIA 포뮬러 원 월드 챔피언십'으로, 국제 스포츠 캘린더에 등록된 여러 그랑프리 경기인 'Competition'들로 구성됩니다. 이 대회에는 드라이버 부문과 제조사 부문의 두 가지 월드 챔피언 타이틀이 있습니다.
----------------------------------------------------------------------------------------------------
[Section] A
[조문] A1.1 | Overview
Q: FIA 포뮬러 원 챔피언십에 대한 상업적 권리는 누가 소유하며, 어떻게 관리되나요?
A: 챔피언십의 상업적 권리는 독점적으로 FIA가 소유하고 있으며, FIA가 지정한 상업 권리 보유자에게 이 권리를 독점적으로 활용할 수 있도록 허가한 상태입니다.
----------------------------------------------------------------------------------------------------
[Section] A
[조문] A1.1 | Overview
Q: FIA 포뮬러 원 규정은 몇 개의 주요 섹션으로 나뉘며, 각 섹션의 주된 내용은 무엇인가요?
A: FIA F1 규정은 총 6개 주요 섹션으로 구성되어 있습니다: 총괄 규정, 스포츠 규정, 기술 규정, 팀 재정 규정, 엔진 제조사 재정 규정, 운영 규정입니다.
----------------------------------------------------------------------------------------------------
[Section] A
[조문] A1.1 | Overview
Q: FIA F1 문서는 어떤 종류가 있으며, 각각